In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../'))

import numpy as np
import tensorflow as tf
import random, json

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED); random.seed(SEED)

In [ ]:
from src.pipeline.captioning_scratch import ImageCaptionerScratch
from src.pipeline.captioning_keras import ImageCaptionerKeras
from src.evaluation.metrics import compute_bleu4, compute_meteor, timed_predict_from_features

VOCAB_PATH          = '../../data/captions/vocab.json'
TEST_CAPTIONS_PATH  = '../../data/captions/test_captions.json'
TEST_FEATURES_PATH  = '../../data/features/test_features.npy'

with open(TEST_CAPTIONS_PATH) as f:
    test_data = json.load(f)

test_image_names = list(test_data.keys())
test_references  = [test_data[name] for name in test_image_names]

test_features = np.load(TEST_FEATURES_PATH, allow_pickle=True).item()
if isinstance(test_features, dict):
    test_feat_matrix = np.stack([test_features[n] for n in test_image_names])
else:
    test_feat_matrix = test_features

print(f'Test set: {len(test_image_names)} images')

In [ ]:
BEST_RNN_WEIGHTS  = '../../weights/rnn_lstm/rnn_preinject_L?_H???.h5'   
BEST_LSTM_WEIGHTS = '../../weights/rnn_lstm/lstm_preinject_L?_H???.h5'  

BEST_RNN_LABEL  = 'rnn_Xlayer_YYY'   
BEST_LSTM_LABEL = 'lstm_Xlayer_YYY'

In [ ]:
def evaluate(captioner, feat_matrix, references, max_len=20):
    captions, total, avg = timed_predict_from_features(captioner, feat_matrix, max_len)
    return {
        'bleu4':   compute_bleu4(references, captions),
        'meteor':  compute_meteor(references, captions),
        'avg_time': avg,
        'captions': captions,
    }

# RNN from scratch
rnn_scratch = ImageCaptionerScratch('rnn')
rnn_scratch.load_weights('InceptionV3', BEST_RNN_WEIGHTS, VOCAB_PATH)
res_rnn_scratch = evaluate(rnn_scratch, test_feat_matrix, test_references)
print(f'RNN  Scratch  BLEU-4={res_rnn_scratch["bleu4"]:.4f}  METEOR={res_rnn_scratch["meteor"]:.4f}  avg={res_rnn_scratch["avg_time"]:.3f}s')

# LSTM from scratch
lstm_scratch = ImageCaptionerScratch('lstm')
lstm_scratch.load_weights('InceptionV3', BEST_LSTM_WEIGHTS, VOCAB_PATH)
res_lstm_scratch = evaluate(lstm_scratch, test_feat_matrix, test_references)
print(f'LSTM Scratch  BLEU-4={res_lstm_scratch["bleu4"]:.4f}  METEOR={res_lstm_scratch["meteor"]:.4f}  avg={res_lstm_scratch["avg_time"]:.3f}s')

# RNN Keras
rnn_keras = ImageCaptionerKeras('rnn')
rnn_keras.load_model('InceptionV3', BEST_RNN_WEIGHTS, VOCAB_PATH)
res_rnn_keras = evaluate(rnn_keras, test_feat_matrix, test_references)
print(f'RNN  Keras    BLEU-4={res_rnn_keras["bleu4"]:.4f}  METEOR={res_rnn_keras["meteor"]:.4f}  avg={res_rnn_keras["avg_time"]:.3f}s')

# LSTM Keras
lstm_keras = ImageCaptionerKeras('lstm')
lstm_keras.load_model('InceptionV3', BEST_LSTM_WEIGHTS, VOCAB_PATH)
res_lstm_keras = evaluate(lstm_keras, test_feat_matrix, test_references)
print(f'LSTM Keras    BLEU-4={res_lstm_keras["bleu4"]:.4f}  METEOR={res_lstm_keras["meteor"]:.4f}  avg={res_lstm_keras["avg_time"]:.3f}s')

In [ ]:
import pandas as pd

comparison = [
    {'model': f'RNN-Scratch ({BEST_RNN_LABEL})',   **{k: round(v, 4) for k, v in res_rnn_scratch.items()  if k != 'captions'}},
    {'model': f'RNN-Keras   ({BEST_RNN_LABEL})',   **{k: round(v, 4) for k, v in res_rnn_keras.items()    if k != 'captions'}},
    {'model': f'LSTM-Scratch ({BEST_LSTM_LABEL})', **{k: round(v, 4) for k, v in res_lstm_scratch.items() if k != 'captions'}},
    {'model': f'LSTM-Keras   ({BEST_LSTM_LABEL})', **{k: round(v, 4) for k, v in res_lstm_keras.items()   if k != 'captions'}},
]
df_cmp = pd.DataFrame(comparison)
display(df_cmp)

import json, os
os.makedirs('../../results/tables', exist_ok=True)
with open('../../results/tables/keras_vs_scratch.json', 'w') as f:
    json.dump([{k: v for k, v in r.items() if k != 'captions'} for r in comparison], f, indent=2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns, os

sns.set_theme(style='whitegrid')
os.makedirs('../../results/plots', exist_ok=True)

models  = [r['model'] for r in comparison]
bleu4s  = [r['bleu4']    for r in comparison]
meteors = [r['meteor']   for r in comparison]
times   = [r['avg_time'] for r in comparison]
colors  = ['#4C72B0', '#729FCF', '#DD8452', '#E8A87C']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, vals, title, ylabel in [
    (axes[0], bleu4s,  'BLEU-4',  'Score'),
    (axes[1], meteors, 'METEOR',  'Score'),
    (axes[2], times,   'Avg Inference Time', 'Seconds'),
]:
    bars = ax.bar(models, vals, color=colors)
    ax.set_title(f'{title}: Keras vs Scratch', fontsize=13)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xticklabels(models, rotation=25, ha='right', fontsize=8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../../results/plots/keras_vs_scratch.png', dpi=150, bbox_inches='tight')
plt.show()

## Analisis

**1. Perbedaan score Keras vs from-scratch:**

**2. Waktu inferensi:**